# Embed MS MARCO (8.8M Full Corpus) with all-MiniLM-L6-v2

This notebook is designed to run efficiently on Google Colab with a T4 GPU. It circumvents local download bottlenecks by utilizing Colab's fast network to download the BeIR MS MARCO dataset, computes 384d embeddings using `sentence-transformers`, and saves them directly to an HDF5 file in chunks to prevent CPU RAM crashes.

**Instructions:**
1. Upload this file to Google Colab.
2. Go to **Runtime > Change runtime type** and ensure **T4 GPU** is selected.
3. Run all cells. It will ask for permission to mount your Google Drive.
4. The resulting `msmarco-8.8M-minilm-384d.hdf5` file will be saved directly to your Google Drive!

In [ ]:
# Address PyTorch CUDA memory fragmentation issues
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Mount Google Drive so we can save the massive file directly there
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. Install dependencies
!pip install -q sentence-transformers datasets h5py

In [ ]:
import torch
import h5py
import numpy as np
import gc
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from tqdm.auto import tqdm

# 2. Initialize Model
# We use all-MiniLM-L6-v2 which embeds to 384 dimensions
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
assert device == "cuda", "Please enable a GPU runtime!"

model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
# Enforce max sequence length to prevent long outliers from causing massive VRAM spikes
model.max_seq_length = 256

In [ ]:
# 3. Load the full 8.8 million MS MARCO corpus
print("Downloading and loading BeIR MS MARCO corpus (8.8M)...")
dataset = load_dataset("BeIR/msmarco", "corpus", split="corpus")
print(f"Total passages loaded: {len(dataset):,}")

In [ ]:
# 4. Prepare text data
def get_text(example):
    if example.get('title'):
        return example['title'] + " " + example['text']
    return example['text']

print("Formatting texts for embedding...")
texts = [get_text(ex) for ex in tqdm(dataset, desc="Preparing Texts")]

In [ ]:
# 5. Generate and Stream Embeddings to Disk
# We process in chunks to prevent Colab from running out of CPU RAM and swapping to disk,
# which causes 8-hour slowdowns.
output_file = "/content/drive/MyDrive/msmarco-8.8M-minilm-384d.hdf5"
print(f"Creating/Appending HDF5 file directly at {output_file}...")

N = len(texts)
BATCH_SIZE = 1024  # Reduced from 2048 to prevent CUDA OutOfMemory
CHUNK_SIZE = 500_000

RESUME_INDEX = 6_000_000  # <--- Starting from Chunk 13 (12 * 500k = 6M)

# Open in 'a' mode (append/read-write) so we DON'T overwrite the 3 hours of work!
with h5py.File(output_file, 'a') as f:
    if 'train' not in f:
        # Create an empty dataset of the final size (8.8M x 384)
        dset = f.create_dataset('train', shape=(N, 384), dtype=np.float32)
    else:
        dset = f['train']
        print("Found existing dataset, resuming...")
    
    for i in range(RESUME_INDEX, N, CHUNK_SIZE):
        chunk_texts = texts[i:i + CHUNK_SIZE]
        print(f"\nEncoding chunk {i//CHUNK_SIZE + 1}/{(N//CHUNK_SIZE) + 1} ({len(chunk_texts):,} passages)...")
        
        chunk_emb = model.encode(
            chunk_texts, 
            batch_size=BATCH_SIZE, 
            show_progress_bar=True, 
            convert_to_numpy=True
        )
        
        # Write to disk immediately
        dset[i:i + len(chunk_texts)] = chunk_emb
        
        # Force free memory so Colab doesn't swap to disk
        del chunk_texts
        del chunk_emb
        gc.collect()
        torch.cuda.empty_cache()  # Free PyTorch VRAM to prevent fragmentation

print("\nDone! The file is safely stored in your Google Drive.")